In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

In [2]:
# Load the dataset from CSV file and define column names
df = pd.read_csv('Biometrics_4HZ/labeled_combined_biometric_data_4HZ.csv', low_memory=False)
df.columns = ['acc_x','acc_y','acc_z','bvp','eda', 'hr', 'SPO2','Pulse_Rate', 'motion','temp','Subject','Session','Label']

In [3]:
# Convert columns to numeric if possible
df[['SPO2', 'Pulse_Rate']] = df[['SPO2', 'Pulse_Rate']].apply(pd.to_numeric, errors='coerce')

In [4]:
# Interpolate NaN values for SPO2 and Pulse_rate subject-session wise
df[['SPO2', 'Pulse_Rate']] = df.groupby(['Subject', 'Session'])[['SPO2', 'Pulse_Rate']].apply(
    lambda group: group.interpolate(method='linear', limit_direction='forward')
).reset_index(drop=True)[['SPO2', 'Pulse_Rate']]

### Mixed-effects model for each feature ###

In [ ]:
def map_drowsiness_level(x):
    if x < 4:
        return 'Low'
    elif 4 <= x <= 6:
        return 'Med'
    else:
        return 'High'   

df['DrowsinessLevel'] = df['Label'].apply(map_drowsiness_level)


ndf = df.drop(columns=['Session', 'Label'])

# List of features to analyze (excluding 'Subject' and 'DrowsinessLevel')
features = ['eda', 'temp', 'hr', 'acc_x', 'acc_y', 'acc_z', 'bvp', 'SPO2', 'Pulse_Rate']

# Ensure 'DrowsinessLevel' is treated as a categorical variable
ndf['DrowsinessLevel'] = ndf['DrowsinessLevel'].astype(str)

# Loop through each feature and fit a mixed-effects model
for feature in features:
    print(f"\nMixed-Effects Model for {feature}:")

    # Prepare data for the current feature
    df_feature = ndf[['Subject', 'DrowsinessLevel', feature]].dropna()

    # Fit the mixed-effects model
    try:
        model = smf.mixedlm(f"{feature} ~ DrowsinessLevel", df_feature, groups=df_feature["Subject"])
        result = model.fit()
        print(result.summary())
    except Exception as e:
        print(f"An error occurred for {feature}: {e}")